# Get hold of a built artifact

Two ways in, one object out:

- **point at its folder** -- `Artifact.at(path)`. The manifest in the folder
  says which artifact it is, the folder says where its root is. Any path works
  the same: a local volume, a copy pulled down with `modal volume get`, or
  `/storage/...` on the mounted volume inside a container.
- **write its parameters** -- `Tokenizer(vocab_size=..., ...)`, then
  `.bind(root)`.

`bind` is what turns a named artifact into a usable one. For most artifacts
that's just a check that the files are there; a tokenizer loads its vocab and
merges, so it comes back able to encode.

Nothing below is tokenizer-specific except the class name -- any artifact type
is reached the same way.

In [1]:
from pathlib import Path

from dag.artifact import Artifact
from tokenizers.bpe import Tokenizer  # importing the family is what lets a manifest name it

ROOT = Path(".scratch/demo-volume").resolve()  # a volume, or a copy of one

# 1. by path
tokenizer = Artifact.at(ROOT / "tokenizers/bpe-1000-feeeeefa90")

print(tokenizer)
print(f"{len(tokenizer.vocab)} vocab entries, {len(tokenizer.merges)} merges")

Tokenizer(vocab_size=1000, special_tokens=('<|endoftext|>',), sources=(Source(name='romeojuliet', url='https://www.gutenberg.org/cache/epub/1513/pg1513.txt'),))
1000 vocab entries, 743 merges


In [2]:
line = "But soft, what light through yonder window breaks?<|endoftext|>"

ids = tokenizer.encode(line)
print(ids)
print(repr(tokenizer.decode(ids)))

[487, 380, 102, 116, 44, 542, 711, 285, 114, 854, 296, 111, 814, 262, 520, 310, 756, 97, 485, 63, 999]
'But soft, what light through yonder window breaks?<|endoftext|>'


In [3]:
# 2. by parameters -- the same artifact, so the same object
from sources.artifact import Source

written_out = Tokenizer(
    vocab_size=1000,
    special_tokens=("<|endoftext|>",),
    sources=(Source(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt"),),
).bind(ROOT)

print(written_out == tokenizer, written_out.encode(line) == ids)

# and an artifact that isn't built says so rather than half-working
try:
    Tokenizer(vocab_size=42, special_tokens=(), sources=()).bind(ROOT)
except FileNotFoundError as err:
    print("not built ->", err)

True True
not built -> bpe-42-643d543710 is not built -- missing ['/Users/oguz/Projects/launchpad/.scratch/demo-volume/tokenizers/bpe-42-643d543710/tokenizer.json']
